# Almost every pair is missing

Line the 943 users against the 1682 movies. The full grid has

$$
943 \times 1682 = 1586126
$$

cells. The file fills 100000 of them.

$$
\dfrac{100000}{1586126} = \dfrac{50000}{793063}.
$$

The other $1586126 - 100000 = 1486126$ cells are empty. An empty cell is not a rating of zero. Zero is not even on the scale. The first thing a collaborative filter has to refuse is treating "this person never rated this movie" as "this person hated it."

Every user id from 1 to 943 appears, and every movie id from 1 to 1682 appears at least once. No row and no column of the grid is entirely empty. The sparsest user still has 20 ratings: the release removed anyone below that. The busiest user has 737. The busiest movie has 583.

So the matrix is sparse, and it is not missing an entire person or an entire movie. The holes are inside the rows.


In [2]:
# Locate the MovieLens tables beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "15-Movie Recommendation" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "u.info").exists():
            return direct
    raise FileNotFoundError(name)

def read_lines(name):
    return data_path(name).read_text(encoding="latin-1").splitlines()

def read_ratings(name):
    rows = []
    for line in read_lines(name):
        if not line.strip():
            continue
        user, item, rating, stamp = line.split("\t")
        rows.append((int(user), int(item), int(rating), int(stamp)))
    return rows

ratings = read_ratings("u.data")
print(len(ratings), ratings[0])


100000 (196, 242, 3, 881250949)


In [3]:
# Grid size, filled cells, and the shortest and longest rows.
from fractions import Fraction
from collections import Counter
users = Counter(user for user, _, _, _ in ratings)
movies = Counter(item for _, item, _, _ in ratings)
cells = 943 * 1682
print(cells, len(ratings), cells - len(ratings))
print(Fraction(len(ratings), cells))
print(min(users.values()), max(users.values()), min(movies.values()), max(movies.values()))
assert cells == 1586126
assert cells - 100000 == 1486126
assert Fraction(100000, cells) == Fraction(50000, 793063)
assert set(users) == set(range(1, 944))
assert set(movies) == set(range(1, 1683))
assert min(users.values()) == 20
assert max(users.values()) == 737
assert max(movies.values()) == 583


1586126 100000 1486126
50000/793063
20 737 1 583


## Pitfall

Filling the empty cells with 0 makes *Toy Story* look hated by the 943 minus the people who rated it. Those people did not rate it. A similarity that treats 0 as a score will say that two people agree because both skipped the same movies.

## Summary

The grid is $943 \times 1682 = 1586126$ cells, and $50000/793063$ of it is filled. Every user has at least 20 ratings. An empty cell stays empty.
